# PSELDNets — outdoor_siren_v3 学習（train 60 / val 20）

**難易度軸の本命**: ①サイレンは各クリップ2〜6秒だけ発音（スパース化→誤検出・見逃しが初めて測れる。全フレームの約4〜6割が負例）、②車の実録音がラベルなしの指向性妨害として別軌道を通過（SIR 0〜15 dB）、③拡散ピンク背景雑音（SNR 0〜20 dB、発音区間基準）。学習は毎回、基盤チェックポイントから（実験間の初期条件を揃えるため）
セットアップは 2026-06 に完走実績のある PSELDNets_Colab_v3 と同一レシピ。

## ⚠️ 使用前に必ず確認

1. **ランタイム → ランタイムのタイプを変更 → T4 GPU** を選択
2. **Drive の `MyDrive/PSELDNets_data/` に `dataset_outdoor_siren_v3.zip` をアップロード済み**であること
3. セルを**上から順に**実行（途中でスキップしない）
4. 中断しても再実行すれば続きから学習が再開される（Drive永続化＋自動resume）

---
## 1. GPU 確認

In [ ]:
import torch
assert torch.cuda.is_available(), '⚠️ GPU未接続。ランタイムのタイプを T4 GPU に変更してください。'
print(f'PyTorch : {torch.__version__}')
print(f'GPU     : {torch.cuda.get_device_name(0)}')
print(f'VRAM    : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

## 2. Drive マウントとパス設定

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# ==== 設定（必要ならここだけ書き換える） ====
DRIVE_DATA = '/content/drive/MyDrive/PSELDNets_data'   # zip の置き場所
DRIVE_LOGS = '/content/drive/MyDrive/PSELDNets_logs'   # 学習ログ・ckpt の永続化先
DRIVE_CKPT = '/content/drive/MyDrive/PSELDNets_ckpts'  # 事前学習ckptのキャッシュ
DATASET    = 'outdoor_siren_v3'
EXP_NAME   = 'outdoor_siren_v3_run1'                             # 固定名（resume用）
ZIP_NAME   = f'dataset_{DATASET}.zip'

import os
for d in [DRIVE_DATA, DRIVE_LOGS, DRIVE_CKPT]:
    os.makedirs(d, exist_ok=True)
ZIP_PATH = f'{DRIVE_DATA}/{ZIP_NAME}'
assert os.path.exists(ZIP_PATH), f'⚠️ {ZIP_PATH} がありません。zip をアップロードしてください。'
print(f'OK: {ZIP_PATH} ({os.path.getsize(ZIP_PATH)/1e6:.0f} MB)')

## 3. リポジトリ clone

In [ ]:
import os

REPO = '/content/PSELDNets'

if not os.path.exists(f'{REPO}/src'):
    !git clone https://github.com/Jinbo-Hu/PSELDNets {REPO}
else:
    print(f'既にあります: {REPO}')

os.chdir(REPO)
print(f'CWD: {os.getcwd()}')

## 4. パッケージインストール

`numpy` / `h5py` / `scipy` / `torch` は Colab に最初から入っています。
ここでは**触らず**、不足しているものだけ追加します（v3 と同一・再起動不要）。

In [ ]:
!pip install -q \
    librosa \
    soundfile \
    lightning==2.2.1 \
    hydra-core==1.3.2 \
    hydra-colorlog==1.2.0 \
    hydra-joblib-launcher==1.2.0 \
    torchmetrics==1.3.1

import numpy, lightning, torchmetrics, librosa
print(f'numpy {numpy.__version__} / lightning {lightning.__version__} / '
      f'torchmetrics {torchmetrics.__version__} / librosa {librosa.__version__}')

## 5. 事前学習チェックポイント（Drive キャッシュ → なければ HF から）

In [ ]:
import shutil

os.makedirs('ckpts', exist_ok=True)
CKPT = 'ckpts/mACCDOA-HTSAT-0.567.ckpt'
CACHE = f'{DRIVE_CKPT}/mACCDOA-HTSAT-0.567.ckpt'

if not os.path.exists(CKPT):
    if os.path.exists(CACHE):
        print('Drive キャッシュからコピー...')
        shutil.copy(CACHE, CKPT)
    else:
        print('HuggingFace からダウンロード...')
        from huggingface_hub import hf_hub_download
        src = hf_hub_download(repo_id='Jinbo-HU/PSELDNets',
                              filename='model/mACCDOA-HTSAT-0.567.ckpt',
                              repo_type='dataset')
        shutil.copy(src, CKPT)
        shutil.copy(CKPT, CACHE)   # 次回用に Drive へキャッシュ
print(f'OK: {CKPT} ({os.path.getsize(CKPT)/1e6:.0f} MB)')

## 6. データセット展開

zip は `datasets/...` 構成なのでリポジトリ直下で解凍するだけ。
クラス辞書 `cls_indices_train.tsv`（屋外10クラス, Siren=class 4）も同梱。

In [ ]:
import zipfile

if not os.path.exists(f'datasets/{DATASET}/foa'):
    with zipfile.ZipFile(ZIP_PATH) as z:
        z.extractall('.')
    print('解凍完了')

n_foa = len(os.listdir(f'datasets/{DATASET}/foa'))
n_meta = len(os.listdir(f'datasets/{DATASET}/metadata'))
n_cls = len(open('datasets/cls_indices_train.tsv').readlines())
print(f'foa: {n_foa} / metadata: {n_meta} / classes: {n_cls}')
assert n_foa == 80 and n_meta == 80 and n_cls == 10, '⚠️ ファイル数が想定と違います'

## 7. 設定ファイル2つを作成（新規追加のみ・リポジトリ既存ファイルは無編集）

In [ ]:
data_yaml = """audio_type: foa
audio_feature: logmelIV
sample_rate: 24000
nfft: 1024
n_mels: 64
hoplen: 240
window: hann

train_chunklen_sec: 10
train_hoplen_sec: 10
test_chunklen_sec: 10
test_hoplen_sec: 10

train_dataset:
  outdoor_siren_v3: [fold1_room1]
valid_dataset:
  outdoor_siren_v3: [fold2_room1]
test_dataset:
  outdoor_siren_v3: [fold2_room1]
"""

exp_yaml = """# @package _global_
defaults:
 - override /data: outdoor_siren_v3.yaml
 - override /loss: multi_accdoa.yaml
 - _self_

task_name: outdoor_siren_v3

model:
  batch_size: 8
  kwargs:
    pretrained_path: ckpts/mACCDOA-HTSAT-0.567.ckpt
    audioset_pretrain: false
  optimizer:
    kwargs: {lr: 0.0003}
  lr_scheduler:
    kwargs: {step_size: 60}

trainer:
  max_epochs: 70
  check_val_every_n_epoch: 5
"""

open('configs/data/outdoor_siren_v3.yaml', 'w').write(data_yaml)
open('configs/experiment/outdoor_siren_v3.yaml', 'w').write(exp_yaml)
print('wrote configs/data/outdoor_siren_v3.yaml')
print('wrote configs/experiment/outdoor_siren_v3.yaml')

## 8. 前処理（ラベル → HDF5、クリップ索引の作成。1分未満）

In [ ]:
IDX = f'_hdf5/data/24000fs/wav/dev/{DATASET}_10sChunklen_10sHoplen_train.csv'
if not os.path.exists(IDX):
    !python src/preproc.py dataset={DATASET}
else:
    print('既に前処理済み')
!head -3 {IDX}

## 9. 実行前チェック

In [ ]:
checks = [
    ('ckpts/mACCDOA-HTSAT-0.567.ckpt',     'チェックポイント'),
    ('datasets/cls_indices_train.tsv',      'クラス辞書 TSV'),
    (f'datasets/{DATASET}/foa',             'FOA データ (80)'),
    (f'datasets/{DATASET}/metadata',        'ラベル CSV (80)'),
    (f'configs/experiment/{DATASET}.yaml',  '実験設定'),
    (IDX,                                   'クリップ索引'),
]
for path, name in checks:
    ok = os.path.exists(path) and (not os.path.isdir(path) or len(os.listdir(path)) > 0)
    print(f'  [{"OK" if ok else "NG"}] {name}')

## 10. 学習（T4 で 15〜30 分見込み）

- ログと ckpt は Drive（`PSELDNets_logs`）に直接書くので、セッションが切れても消えない
- `last.ckpt` があれば自動で続きから再開（固定 experiment_name 方式）
- メモリ不足になったら `model.batch_size=4` に下げて再実行

In [ ]:
LAST = f'{DRIVE_LOGS}/{DATASET}/runs/{EXP_NAME}/checkpoints/last.ckpt'
resume = f'ckpt_path={LAST}' if os.path.exists(LAST) else ''
print('resume:', resume or '(new run)')

!python src/train.py experiment={DATASET} \
    experiment_name={EXP_NAME} \
    paths.log_dir={DRIVE_LOGS} \
    {resume}

## 11. 結果の確認

`val/macro` の ER / F / LE / LR / **SELD_scr** を全エポック分抽出する。
**この val は学習に見せていない20本**（同一生成器の未見シーン）。

**v3の見どころ**: v1/v2は満点に飽和した（負例なし・拡散雑音は方向手がかりを壊さないため）。v3では初めて誤検出・見逃しが可能になったので、**ER/Fが本当の意味を持つ**。満点でも0点でもない中間の値になれば「ものさし」完成。低SNR・低SIRのクリップで悪化するかを inspection.csv と突き合わせて分析する。

In [ ]:
import re

log_path = f'{DRIVE_LOGS}/{DATASET}/runs/{EXP_NAME}/train.log'
lines = [l for l in open(log_path, errors='ignore')
         if 'val/macro' in l or 'train: loss_all' in l]
print(f'--- {log_path} ---')
for l in lines:
    print(re.sub(r'\x1b\[[0-9;]*m', '', l).rstrip())

vals = [l for l in lines if 'val/macro' in l]
if vals:
    print('\n=== 最終 val/macro ===')
    print(re.sub(r'\x1b\[[0-9;]*m', '', vals[-1]).strip())

---
## メモ

- データ生成条件の全記録はローカルの `outdoor_seld_e2e/out/dataset_outdoor_siren_v3/`
  （`inspection.csv`=検品結果, `work/*/scene.json`=各クリップの条件）
- 妨害音（車）はラベルなし＝DCASE2021の「対象クラス外の指向性妨害」方式
- クリーン版・妨害単体の FOA はローカル work/ に保全（要素 on/off 比較は再生成不要）
- SNR/SIR はサイレン発音区間の W チャンネル基準で定義
- 結果を報告するときは ER / F / LE / LR / SELD_scr を併記する